## Basic prompting

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
import os
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage

model = init_chat_model(
    model="deepseek/deepseek-v4-flash-0731",
    model_provider="deepseek",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_KEY"],
)

agent = create_agent(model=model)

question = HumanMessage(content="What's the capital of the moon?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

The Moon doesn’t have a capital — it’s not a country or a political entity. But if you’re looking for the capital **letter**, it’s “M”. 🌙


In [7]:
system_prompt = "You are a science fiction writer, create a capital city at the users request."

scifi_agent = create_agent(
    model=model,
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

The capital of the Moon isn't a sprawling, domed city resting on the grey regolith like a toppled glass snow-globe. That's an old Earth mistake. It's a city you can't see from the surface at all, because it burrows *down*. 

Its name is **Aurelia**, and it is the seat of the Lunar Meridian Compact.

**The Location & Architecture**

Forget the equator. Aurelia is nestled in the spilled ink of the polar dark, specifically inside the rim of the Shackleton Crater at the lunar South Pole. It was built not to *catch* the sun, but to *harness* it. The city hugs the "Peak of Eternal Light" on the crater's ridge, a mountain that basks in perpetual sunlight for nearly a full terrestrial year. 

The city itself is a vertical shaft descending four kilometers into the crater floor, surrounded by a ring of fourteen terraced "petals" — massive, spinning centrifuge rings that provide artificial gravity for the residential and agricultural districts. The shaft, called the Spine, is the administrative a

## Few-shot examples

In [8]:
system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

User: What is the capital of mars?
Scifi Writer: Marsialis

User: What is the capital of Venus?
Scifi Writer: Venusovia

"""

scifi_agent = create_agent(
    model=model,
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

The capital of the Moon is **Selene Prime** — a glittering latticework of dome-cities and light-bridges rising from the ancient dust of Mare Imbrium. Its heart is the **Lunar Spire**, a kilometer-tall needle of regolith-glass and carbon nanotube webbing, where the Council of Earth’s Children convenes under a perpetual Earthshine. Officially, the charter says the capital floats in the terminator’s twilight, forever balanced between day and night, so that no one forgets the Moon holds both faces: one turned toward home, one toward the deep.


## Structured prompts

In [10]:
system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

Please keep to the below structure.

Name: The name of the capital city

Location: Where it is based

Vibe: 2-3 words to describe its vibe

Economy: Main industries

"""

scifi_agent = create_agent(
    model=model,
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

**Name:** Nova Selene

**Location:** The city clings to the perpetual twilight rim of Shackleton Crater at the Lunar South Pole. Its gleaming towers rise from the sunlit peaks, while its deepest industrial sectors descend into the crater's permanent, primordial shadow, where water-ice mines hum in the dark.

**Vibe:** Crystalline, Precipitous, Eternal.

**Economy:** 
- **Helium-3 Refinement:** Nova Selene is the primary exporter of Helium-3 harvested from the lunar regolith, fueling the fusion reactors that power most of the Terra-Luna economic sphere.
- **Micro-Gravity Biotech:** The extreme low-G and cryogenic darkness of the deep crater labs allow for the growth of impossible, perfect protein crystals and genetically engineered compounds that can't be synthesized on Earth.
- **Luxury Low-G Tourism:** The wealthiest citizens of Earth and Mars flock to "The Rim" to experience zero-gravity spas, crater-rim cliff diving with retro-rocket wingsuits, and the celebrated "Earthrise" observa

## Structured output

In [11]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pydantic import BaseModel

class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

agent = create_agent(
    model=model,
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
    response_format=CapitalInfo
)

question = HumanMessage(content="What is the capital of The Moon?")

response = agent.invoke(
    {"messages": [question]}
)

response["structured_response"]

CapitalInfo(name='Lunara', location='Mare Serenitatis, inside the crater Bessel, domed beneath a transparent diamond-carbon canopy', vibe='Silvery and serene, swaddled in perpetual blue-hour twilight, with fountains of mist floating in the low gravity and streets carved into the regolith', economy='Lunar helium-3 mining and low-gravity hydroponics')

In [17]:
response['structured_response']


CapitalInfo(name='Lunara', location='Mare Serenitatis, inside the crater Bessel, domed beneath a transparent diamond-carbon canopy', vibe='Silvery and serene, swaddled in perpetual blue-hour twilight, with fountains of mist floating in the low gravity and streets carved into the regolith', economy='Lunar helium-3 mining and low-gravity hydroponics')

In [18]:
response["structured_response"].name

'Lunara'

In [19]:
capital_info = response["structured_response"]

capital_name = capital_info.name
capital_location = capital_info.location

print(f"{capital_name} is a city located at {capital_location}")

Lunara is a city located at Mare Serenitatis, inside the crater Bessel, domed beneath a transparent diamond-carbon canopy
